In [1]:
import os
from datasets import load_dataset, Audio
import numpy as np
import soundfile as sf
import whisper

/home/andresmtr/miniconda3/envs/PruebaBRM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Crear carpeta para audios y transcripciones
os.makedirs("audios", exist_ok=True)
os.makedirs("transcripciones", exist_ok=True)

In [3]:
# Cargar el dataset de Hugging Face
dataset = load_dataset("charris/hubert_process_filter_spotify", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [4]:
dataset

Dataset({
    features: ['audio', 'transcription', 'input_values', 'input_length', 'labels'],
    num_rows: 1183
})

In [5]:
print("\n✅ Tipo de columna 'audio':", type(dataset[0]["audio"]))


✅ Tipo de columna 'audio': <class 'dict'>


In [6]:
# Función para guardar audio desde numpy
def guardar_audio(array, path, sample_rate=16000):
    sf.write(path, array, sample_rate)

In [7]:

# Función para transcribir y guardar resultado
def transcribir_audio(ruta_audio, modelo, carpeta_salida="transcripciones"):
    resultado = modelo.transcribe(ruta_audio)
    texto = resultado["text"]
    nombre_archivo = os.path.splitext(os.path.basename(ruta_audio))[0]
    ruta_txt = os.path.join(carpeta_salida, f"{nombre_archivo}.txt")
    with open(ruta_txt, "w", encoding="utf-8") as f:
        f.write(texto)
    print(f"Transcripción guardada: {ruta_txt}")
    return texto

In [ ]:
audio_paths = []
for i in range(5):
    audio_info = dataset[i]["audio"]  
    audio_array = audio_info["array"]
    sample_rate = audio_info["sampling_rate"]
    audio_path = f"audios/audio_{i}.wav"
    guardar_audio(audio_array, audio_path, sample_rate)
    audio_paths.append(audio_path)

# Cargar modelo Whisper
modelo_whisper = whisper.load_model("medium")

100%|█████████████████████████████████████| 1.42G/1.42G [01:43<00:00, 14.7MiB/s]
/home/andresmtr/miniconda3/envs/PruebaBRM/lib/python3.12/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for a

In [ ]:
# Transcribir cada audio
for ruta in audio_paths:
    print(f"Sonido Transcribiendo: {ruta}")
    transcribir_audio(ruta, modelo_whisper)

Sonido Transcribiendo: audios/audio_0.wav
Transcripción guardada: transcripciones/audio_0.txt
Sonido Transcribiendo: audios/audio_1.wav
